# 05 — Query-based detector: RT-DETR (Ultralytics)

RT-DETR-L (COCO weights). The decoder refines a fixed set of object queries; each query predicts
one box or "no object". During training, Hungarian (bipartite) matching assigns **one** query to each
GT box, so duplicates are penalised by the loss and no NMS is needed. The number of queries is
also the maximum number of detections per frame.

| Run | Change | Why |
|---|---|---|
| `rtdetr_base` | 300 queries, input 1280 | baseline |
| `rtdetr_q500` | **500 queries** | up to 180 objects per frame: more queries than the densest scenes need |
| `rtdetr_frozen` | backbone frozen (lr = 0 for the backbone, transformer trained) | separate learning rates for backbone and transformer, extreme case; faster epochs |

Same budget idea as the CNN models. Approximate time on a T4 with `FAST=True`: ~0.5 h per run, ~1.7 h in total (~4 h without FAST).

## 0. Setup
Works on **Kaggle** (add the competition as input, GPU, Internet on), **Google Colab** (GPU T4,
Kaggle token as Colab secret `KAGGLE_API_TOKEN`, or `kaggle.json` in the root of Google Drive) and **a laptop** (Kaggle token in `<repo>/.kaggle_token`, git-ignored;
data and results go to `~/vdet_data` and `~/vdet_work`; without a GPU use it only with `SMOKE = True`).
On Colab all runs are saved to `MyDrive/vdet`, so after a disconnect just run the notebook again:
finished runs are skipped and interrupted runs continue from the last epoch.

In [ ]:
# ---- Settings ----
SMOKE = False   # True: tiny run on a few images, only to check that the notebook works end to end
FAST = True     # True: every 20th train frame and fewer epochs (~2x faster); False: full protocol
REPO = "https://github.com/LidaSokha03/AI_HW2.git"   # shared code is in src/ of this repository

In [ ]:
import os
import sys
import subprocess

if "google.colab" in sys.modules:  # Colab: keep runs and results on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'ultralytics==8.4.207', 'pycocotools', 'kaggle>=2.0'], check=True)

# Shared code (src/vdet.py, src/tvdet.py): local copy if present, otherwise cloned from GitHub
CANDIDATES = ["src", "../src", ".", "notebooks"]
SRC = os.environ.get("VDET_SRC") or next((p for p in CANDIDATES if os.path.exists(f"{p}/vdet.py")), None)
if SRC is None:
    if os.path.exists("/tmp/AI_HW2"):
        subprocess.run(["git", "-C", "/tmp/AI_HW2", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/tmp/AI_HW2"], check=True)
    SRC = next(f"/tmp/AI_HW2/{p}" for p in ["src", "notebooks"] if os.path.exists(f"/tmp/AI_HW2/{p}/vdet.py"))
sys.path.insert(0, os.path.abspath(SRC))

import vdet

# Same frame subset for every model (see the protocol below). First run: ~5 min to prepare.
TRAIN_STEP = 20 if FAST else 10
S = vdet.setup(train_step=TRAIN_STEP, val_step=5, max_side=1600, smoke=SMOKE)
print({k: S[k] for k in ["platform", "work", "prep"]})

In [ ]:
import os
import yaml
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import RTDETR

NB = "05_rtdetr"
GT = f"{S['prep']}/val_gt.json"
VAL = vdet.val_items(S)[:20] if SMOKE else vdet.val_items(S)
TEST = vdet.test_items(S)[:30] if SMOKE else vdet.test_items(S)
FP16 = vdet.fp16_args()
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500", "ms_per_img", "FPS"]

### Common protocol (identical in all model notebooks)
| Item | Value |
|---|---|
| Split | given train / val split, which is already **by video** (56 / 7 videos, no leakage) |
| Frames | every 20th train frame (≈1,200, `FAST=True`) or every 10th (≈2,400) and every 5th val frame (≈570): neighbouring frames are near-duplicates |
| Resolution | frames stored with longer side ≤ 1600 px; model input longer side **1280 px** unless the experiment changes it |
| Targets | `iscrowd=1` boxes (ignored regions, "others") are removed from training targets |
| Metric | COCO mAP@[.5:.95], AP50, AP_small… with `pycocotools` on val at **original resolution**, ignored regions as `iscrowd`, ≤ 500 detections per frame (same as the leaderboard) |
| Speed | ms per frame on the same GPU, batch 1, including image reading and NMS |
| Budget | ≈ 0.5 GPU-hour (T4) per run with `FAST=True`, ≈ 1 hour without |
| Seed | 42 |

### Training settings
AdamW, lr 1e-4, weight decay 1e-4, 1 warm-up epoch, 6 epochs (10 without FAST), batch 4, default augmentation.
The 500-query model is built from a copy of `rtdetr-l.yaml` with `nq=500`; all weights are loaded
from the COCO checkpoint (the queries are selected from encoder features, so their number does not
change the weight shapes).

In [ ]:
EPOCHS = 1 if SMOKE else (6 if FAST else 10)

# rtdetr-l.yaml with 500 queries: RTDETRDecoder args = [nc, hidden_dim, num_queries]
cfg = yaml.safe_load(open(os.path.join(os.path.dirname(ultralytics.__file__), "cfg/models/rt-detr/rtdetr-l.yaml")))
cfg["head"][-1][3] = ["nc", 256, 500]
Q500_YAML = f"{S['work']}/rtdetr-l-q500.yaml"
yaml.safe_dump(cfg, open(Q500_YAML, "w"), sort_keys=False)

COMMON = dict(epochs=EPOCHS, optimizer="AdamW", lr0=1e-4, weight_decay=1e-4, warmup_epochs=1,
              warmup_bias_lr=0.0, close_mosaic=min(2, EPOCHS), cache=True, deterministic=False,
              fraction=0.05 if SMOKE else 1.0)
RUNS = {
    "rtdetr_base":   dict(model_src="rtdetr-l.pt", imgsz=1280, batch=4),
    "rtdetr_q500":   dict(model_src=Q500_YAML, load_weights="rtdetr-l.pt", imgsz=1280, batch=4),
    "rtdetr_frozen": dict(model_src="rtdetr-l.pt", imgsz=1280, batch=4, freeze=10),  # layers 0-9 = backbone
}
results, val_preds = {}, {}


def run_rtdetr(name):
    """Train (or reuse) one run, evaluate it with the shared COCO evaluator and log the result."""
    cfg = dict(RUNS[name])
    src, imgsz, batch, load = cfg.pop("model_src"), cfg.pop("imgsz"), cfg.pop("batch"), cfg.pop("load_weights", None)
    run_dir = vdet.ultra_train(S, name, src, imgsz=imgsz, batch=batch, family="rtdetr",
                               load_weights=load, **COMMON, **cfg)
    model = RTDETR(f"{run_dir}/weights/best.pt")
    preds = vdet.ultra_predict(model, VAL, imgsz=imgsz, batch=4)
    metrics = vdet.coco_eval(GT, preds, per_class=True)
    metrics.update(vdet.measure_speed(
        lambda p: model.predict(p, imgsz=imgsz, conf=0.001, verbose=False, **FP16),
        [it["path"] for it in VAL], n=10 if SMOKE else 100))
    vdet.log_result(S, NB, name, metrics, imgsz=imgsz, epochs=EPOCHS)
    results[name], val_preds[name] = metrics, preds
    return pd.DataFrame([metrics], index=[name])[COLS]

## 1. Baseline: 300 queries

In [ ]:
run_rtdetr("rtdetr_base")

## 2. Experiment 1: 500 queries

In [ ]:
run_rtdetr("rtdetr_q500")

## 3. Experiment 2: frozen backbone

In [ ]:
run_rtdetr("rtdetr_frozen")

## 4. Comparison

In [ ]:
table = vdet.results_table(results)
display(table[COLS])
vdet.plot_size_bars(table, "RT-DETR: AP by object size", save=f"{S['work']}/figs/rtdetr_ap_by_size.png")
print("AP per class:")
display(table[[f"AP_{c}" for c in vdet.CLASS_NAMES]].astype(float).round(3))

## 5. Convergence vs a CNN detector with the same budget
Validation mAP (Ultralytics evaluator, same for both) over **training time**. The YOLO curve is
read from the `04_yolo` outputs (on Kaggle: add that notebook's output as input).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
curves = {name: f"{S['work']}/runs/{name}/results.csv" for name in RUNS}
yolo_csv = vdet.find_file(S, "runs/yolo_base/results.csv")
if yolo_csv:
    curves["yolo_base (CNN)"] = yolo_csv
else:
    print("yolo_base results not found - run 04_yolo first to compare")
for name, path in curves.items():
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    t = df["time"] / 3600 if "time" in df else df["epoch"]
    axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], marker="o", label=name)
    axes[1].plot(t, df["metrics/mAP50-95(B)"], marker="o", label=name)
axes[0].set_xlabel("epoch"); axes[1].set_xlabel("training time, hours")
for ax in axes:
    ax.set_ylabel("val mAP50-95"); ax.legend()
plt.tight_layout(); plt.savefig(f"{S['work']}/figs/rtdetr_vs_cnn.png", dpi=110); plt.show()

## 6. Queries vs scene density
How many objects are in a val frame compared with the number of queries, and how many confident
detections each model actually returns.

In [ ]:
import json
gt = json.load(open(GT))
n_obj = pd.Series([a["image_id"] for a in gt["annotations"] if a["iscrowd"] == 0]).value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(n_obj.values, bins=40, alpha=0.6, label="GT objects per frame")
for name in results:
    n_det = pd.Series([p["image_id"] for p in val_preds[name] if p["score"] >= 0.3]).value_counts()
    ax.hist(n_det.values, bins=40, alpha=0.4, label=f"{name}: detections with score >= 0.3")
for q in (300, 500):
    ax.axvline(q, color="k", ls="--", lw=1)
ax.set_xlabel("objects per frame"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{S['work']}/figs/rtdetr_queries.png", dpi=110); plt.show()
print("max GT objects in a val frame:", n_obj.max())

## 7. Errors of the best run

In [ ]:
best = table["mAP"].astype(float).idxmax()
print("best run:", best)
miss = vdet.error_breakdown(S, val_preds[best])
ids = vdet.worst_frames(S, val_preds[best], k=2)
vdet.plot_errors(S, val_preds[best], ids, save=f"{S['work']}/figs/rtdetr_errors.png")

## 8. Submission with the best run

In [ ]:
model = RTDETR(f"{S['work']}/runs/{best}/weights/best.pt")
vdet.write_submission(S, vdet.ultra_predict(model, TEST, imgsz=1280, batch=4), best)